# 리포트 04 — 검출기: CFAR 를 경험 Pfa 로 교정했다

> ### 한 일
> **운용 형상의 검출 사슬에서 경험적 오경보율을 GPU 몬테카를로로 측정하고, 세 파형의 CFAR 문턱을 그 측정값에 맞춰 교정했다.**

### 결과
1. GPU 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ 동안 파형·모드마다 거리-도플러 맵 10,000 ⟨outputs/verify_cfar.json : meta.n_maps_chain⟩장을 돌려 경험 Pfa 를 측정했다.
2. 이상적 백색 맵 500,000 ⟨outputs/verify_cfar.json : meta.n_maps_white⟩장(셀 564,000,000 ⟨outputs/verify_cfar.json : white.48x24.rows[89].cells⟩개)에서 경험/명목 = 0.997 ⟨outputs/verify_cfar.json : white.48x24.rows[89].ratio⟩ — CFAR 구현의 눈금을 먼저 확정했다.
3. 운용 형상(CPI 프레임 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩ · `g2x2_t6x6` · 0-도플러 마스크 1 ⟨outputs/verify_cfar.json : meta.zd_mask_operational⟩빈)에서 명목 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 를 주면 WiFi 1.53 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.op.rows[89].ratio⟩배 · LTE 2.66 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.op.rows[89].ratio⟩배 · 5G 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배로 울린다.
4. 그 형상의 교정표를 만들었다 — 경험 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 를 얻는 명목값은 WiFi 6.27e-05 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ · LTE 2.90e-05 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ · 5G 6.46e-05 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ 다. `src/experiment_detection.py:358` 과 `src/experiment_x410.py:175` 가 `src/passive_process.py:283` 을 거쳐 그 표를 읽는다.
5. 배율의 원인은 slow-time Hann 창이 만드는 도플러축 셀 상관이다 — rect 창으로 두면 5G 가 0.96 ⟨outputs/verify_cfar.json : control_rect_window_NR100.op.rows[89].ratio⟩배로 내려온다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 경험 Pfa | 파형·명목값마다 거리-도플러 맵 10,000 ⟨outputs/verify_cfar.json : meta.n_maps_chain⟩장에 CA-CFAR 를 걸어 오경보 셀을 세었다 (GPU, 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩) |
| 문턱 상수 | CA-CFAR α 를 이론식과 대조 — 상대오차 7.6e-16 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.rel_err⟩ |
| 배율의 원인 | 대조군 2종 — slow-time Hann 제거(도플러축), 백색화 정합필터(거리축) |
| 운용 형상 교정표 | 측정한 명목–경험 곡선을 역보간. 측정 구간 안의 점만 남긴다. 자유공간 기하는 형상이 달라 `src/freespace_detect.py:711` 이 거기서 다시 잰다 |
| ECA 소거 깊이 | 탭 수 1~96 스윕 × (직접파만 / 측정된 다중경로 포함) 두 조건 |
| 분해능 · 관측가능성 | Fisher 정보행렬의 랭크와 CRLB 를 기하에서 계산 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_cfar.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_eca.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_observability.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/make_report04_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_cfar.json`, `outputs/verify_eca.json`, `outputs/verify_observability.json` |
| 소요 | CFAR 측정이 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ (GPU 1장). ECA · 관측가능성 · 그림은 각각 수 분. |
| 비고 | 맵 수는 `--maps` / `--white` 로 줄인다. 줄이면 신뢰구간이 넓어진다. |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 리포트 03 | 세 조명원(WiFi · LTE · 5G NR)의 대역폭 · 기준신호 · 점유 모드 |
| 리포트 01 | 게재 선행 census — 각 논문이 표적 산란을 어떻게 다뤘는가 |

<!--pk:paper_map {"kind": "paper_map", "sections": ["IV. Detection Chain"], "claim": "세 파형의 CFAR 문턱을 GPU 몬테카를로로 측정한 경험 오경보율에 맞춰 교정해, 세 조명원이 같은 실제 오경보율 위에서 비교되게 했다.", "evidence": ["그림 1", "그림 4", "그림 5", "§3 운용 형상 교정표", "outputs/verify_cfar.json:chain.NR100.dpi_eca.calib_op_mask1.points", "outputs/verify_cfar.json:alpha_audit.g2x2_t6x6.rel_err", "outputs/verify_eca.json:S1_depth_vs_taps", "outputs/prior_census.json:counts.zero_cfar_and_falsealarm"], "qualifications": ["교정 배율은 형상이 정한다 — 운용 창과 넓은 창의 값이 달라 형상마다 다시 잰다(§3)", "측정 구간은 명목 1e-06 ⟨outputs/verify_cfar.json : meta.pfa_nominal[8]⟩ ~ 1e-02 ⟨outputs/verify_cfar.json : meta.pfa_nominal[0]⟩ 다. 그 밖의 운용점은 외삽으로 표시된다"], "report": "report04_detector"}-->
> **논문 대응** · **IV. Detection Chain**
>
> 주장 — 세 파형의 CFAR 문턱을 GPU 몬테카를로로 측정한 경험 오경보율에 맞춰 교정해, 세 조명원이 같은 실제 오경보율 위에서 비교되게 했다.
> 근거 — 그림 1 · 그림 4 · 그림 5 · §3 운용 형상 교정표 · `outputs/verify_cfar.json:chain.NR100.dpi_eca.calib_op_mask1.points` · `outputs/verify_cfar.json:alpha_audit.g2x2_t6x6.rel_err` · `outputs/verify_eca.json:S1_depth_vs_taps` · `outputs/prior_census.json:counts.zero_cfar_and_falsealarm`
> 단서 — 교정 배율은 형상이 정한다 — 운용 창과 넓은 창의 값이 달라 형상마다 다시 잰다(§3) · 측정 구간은 명목 1e-06 ⟨outputs/verify_cfar.json : meta.pfa_nominal[8]⟩ ~ 1e-02 ⟨outputs/verify_cfar.json : meta.pfa_nominal[0]⟩ 다. 그 밖의 운용점은 외삽으로 표시된다

---

## §1. 사슬 — 수신 신호가 판정이 되기까지

패시브 검출은 네 단계다. 각 단계는 앞 단계의 잔류물을 물려받는다.

| 단계 | 하는 일 | 코드 |
|---|---|---|
| 1. 수신 | 서베일런스 + 레퍼런스 2채널 | `src/passive_process.py:42` |
| 2. ECA | 직접파를 서베일런스에서 투영 제거 | `src/passive_process.py:93,124` |
| 3. 거리-도플러(CAF) | 레퍼런스와 지연 · 도플러 상관 | `src/passive_process.py:133` |
| 4. CA-CFAR | 이웃 셀로 문턱을 세우고 판정 | `src/passive_process.py:153` |

직접파는 수신단에서 가장 큰 신호다. 그 크기가 DNR 이고, 2단계가 지울 대상이다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report04_f1_chain.png", "figure_no": "1", "question": "수신 신호는 어떤 단계를 거쳐 검출 판정이 되는가?", "paper_caption": "Passive bistatic detection chain applied identically to all three illuminators: a least-squares ECA projection removes the direct path from the surveillance channel, a cross-ambiguity function maps delay against Doppler, and a 2D CA-CFAR declares detections at a threshold calibrated against the measured empirical false-alarm rate.", "vector_pdf": "outputs/figures/report04_f1_chain.pdf", "report": "report04_detector"}-->
![report04_f1_chain.png](outputs/figures/report04_f1_chain.png)

**그림 1.** 수신 신호는 어떤 단계를 거쳐 검출 판정이 되는가?

### 사슬의 형상 — 파형이 정하는 것

거리 빈 수와 ECA 탭 수는 파형이 정한다. 도플러 빈은 세 파형 모두 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩개다(CPI 당 프레임 수).

| 파형 | DNR | ECA 탭 | 거리 빈 | PRF | Δf_d |
|---|---|---|---|---|---|
| WiFi 80MHz | 43.0 dB | 24 | 16 | 1000 Hz | 20.83 Hz |
| LTE 20MHz | 60.0 dB | 14 | 6 | 1000 Hz | 20.83 Hz |
| 5G NR 100MHz | 48.9 dB | 32 | 24 | 2000 Hz | 41.67 Hz |

출처 ⟨outputs/verify_eca.json : meta.setups⟩

## §2. ECA — 직접파를 얼마나 지우고, 무엇을 대가로 내는가

탭을 늘리면 소거가 깊어지다가 환경이 정한 바닥에서 멈춘다. 직접파만 든 신호에 같은 소거기를 걸면 float64 한계까지 내려가고, 측정된 다중경로를 넣으면 오른쪽 값에서 포화한다.

| 파형 | 직접파만 | 직접파 + 다중경로(포화) |
|---|---|---|
| WiFi | 202.7 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[1].rows[12].depth_dpi_db⟩ | 33.0 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[1].rows[12].depth_full_db⟩ |
| LTE | 219.9 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[2].rows[12].depth_dpi_db⟩ | 41.1 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[2].rows[12].depth_full_db⟩ |
| 5G | 232.3 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[0].rows[12].depth_dpi_db⟩ | 56.1 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[0].rows[12].depth_full_db⟩ |

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report04_f2_eca_depth.png", "figure_no": "2", "question": "ECA 소거 깊이의 바닥을 정하는 것은 무엇인가?", "paper_caption": "ECA cancellation depth saturates at a floor set by the measured multipath environment rather than by the number of taps, and that floor differs by more than twenty decibels across the three waveforms.", "vector_pdf": "outputs/figures/report04_f2_eca_depth.pdf", "report": "report04_detector"}-->
![report04_f2_eca_depth.png](outputs/figures/report04_f2_eca_depth.png)

**그림 2.** ECA 소거 깊이의 바닥을 정하는 것은 무엇인가?

### ECA 의 대가 — 0-도플러 노치

ECA 는 지연만 다른 성분을 함께 지운다. 3 dB 손실 지점은 f_d/Δf_d = 0.596 ⟨outputs/verify_eca.json : S4_target_loss[1].fd_3db_over_dfd⟩ 이고, 세 파형이 같다. 속도 문턱은 λ 가 가른다.

CPI 프레임 48 ⟨outputs/verify_eca.json : S4_target_loss[4].M⟩개에서 WiFi 0.39 m/s ⟨outputs/verify_eca.json : S4_target_loss[4].v_3db_ms⟩ · LTE 1.10 m/s ⟨outputs/verify_eca.json : S4_target_loss[7].v_3db_ms⟩ · 5G 1.16 m/s ⟨outputs/verify_eca.json : S4_target_loss[1].v_3db_ms⟩ 아래가 노치 안에 들어간다(그림 3b).

정적 산란체는 ECA 뒤에서 죽은 파라미터다 — 클러터를 100 ⟨outputs/verify_eca.json : S5_clutter_dead.sweep[3].scale⟩배까지 키워도 SCR 변화폭은 3.5e-09 dB ⟨outputs/verify_eca.json : S5_clutter_dead.scr_span_db⟩ 다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report04_f3_eca_notch.png", "figure_no": "3", "question": "ECA 가 클러터와 함께 지우는 표적의 속도는 얼마인가?", "paper_caption": "The ECA zero-Doppler notch removes target energy inside one Doppler bin for all three waveforms, and the resulting minimum detectable radial speed is set by the wavelength and by the coherent processing interval.", "vector_pdf": "outputs/figures/report04_f3_eca_notch.pdf", "report": "report04_detector"}-->
![report04_f3_eca_notch.png](outputs/figures/report04_f3_eca_notch.png)

**그림 3.** ECA 가 클러터와 함께 지우는 표적의 속도는 얼마인가?

## §3. CFAR 교정 — 운용 형상에서 경험 Pfa 를 재고 문턱을 그 값에 맞췄다

⭐ 오경보율을 명목값과 대조하려면 같은 배경을 수만 번 다시 만들어 세어야 한다. 통제 시뮬레이션이 그 일을 한다. 실외 실측은 배경을 주어진 대로 받는다.

이 절의 모든 수는 **운용 형상** 하나에서 나온다 — DPI+ECA · 운용 거리창(§1 표) · CPI 프레임 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩ · 훈련창 `g2x2_t6x6` · 0-도플러 마스크 1 ⟨outputs/verify_cfar.json : meta.zd_mask_operational⟩빈.

선행 census 16 ⟨outputs/prior_census.json : meta.n_papers⟩편 · 전문 198 ⟨outputs/prior_census.json : counts.total_pages⟩쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 ⟨outputs/prior_census.json : counts.zero_cfar_and_falsealarm⟩편이고, 검출을 주장한 논문은 1 ⟨outputs/prior_census.json : counts.claims_detection⟩편이다. OpenISAC(`arXiv:2601.03535v2`, preprint)은 전문 16 ⟨outputs/prior_census.json : papers[14].pages⟩쪽에서 `CFAR` 0 ⟨outputs/prior_census.json : papers[14].terms.cfar⟩회 · `false alarm` 0 ⟨outputs/prior_census.json : papers[14].terms.false_alarm⟩회 · `detection probability` 0 ⟨outputs/prior_census.json : papers[14].terms.detection_probability⟩회다.

검출기 구현의 눈금부터 확정했다 — 문턱 상수는 이론값과 상대오차 7.6e-16 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.rel_err⟩ 안에서 같고, 잡음 추정/실제 전력 = 1.000 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.noise_est_over_power⟩ 다. 이상적 백색 맵 500,000 ⟨outputs/verify_cfar.json : meta.n_maps_white⟩장에서 경험/명목 = 0.997 ⟨outputs/verify_cfar.json : white.48x24.rows[89].ratio⟩ 다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report04_f4_pfa.png", "figure_no": "4", "question": "명목 Pfa 를 요구하면 실제로는 몇 배가 울리는가?", "paper_caption": "The empirical false-alarm rate measured over 10,000 range-Doppler maps per waveform exceeds the nominal rate by a waveform-dependent factor, so the CFAR threshold is calibrated before the three illuminators are compared at one false-alarm rate.", "vector_pdf": "outputs/figures/report04_f4_pfa.pdf", "report": "report04_detector"}-->
![report04_f4_pfa.png](outputs/figures/report04_f4_pfa.png)

**그림 4.** 명목 Pfa 를 요구하면 실제로는 몇 배가 울리는가?

### 운용 형상 교정표 — 목표 경험 Pfa 에 필요한 명목값

운용 명목값은 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 다. 왼쪽 열이 그 값에서 측정된 배율이고, 오른쪽 열이 교정된 명목값이다.

| 파형 | 명목 1e-4 에서 경험/명목 | 경험 1e-4 를 얻을 명목 Pfa |
|---|---|---|
| WiFi | 1.53 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.op.rows[89].ratio⟩배 | 6.27e-05 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ |
| LTE | 2.66 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.op.rows[89].ratio⟩배 | 2.90e-05 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ |
| 5G | 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배 | 6.46e-05 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ |

세 파형의 배율이 서로 다르다. 교정이 셋을 같은 실제 오경보율 위에 올리고, 05편 §3 의 모드별 필요 SNR 표가 그 위에서 선다.

`src/passive_process.py:283` 이 이 JSON 을 읽고, `pfa_nominal_for()`(`src/passive_process.py:338`)가 파형별 명목값을 돌려준다.

In [ ]:
# 교정표를 실제로 소비하는 지점 — src/passive_process.py:283,338
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from passive_process import pfa_nominal_for

for std in ('wifi', 'lte', 'nr'):
    print(f'{std:4s}  경험 1e-4 목표 → 명목 {pfa_nominal_for(std, 1e-4):.3e}')

### 원인 — 셀 상관

CA-CFAR 는 훈련셀이 서로 독립이라고 가정한다. 사슬은 slow-time Hann 창으로 도플러축 셀을 묶는다 — 대조군이 그 항을 원인으로 확정한다(5G NR, 명목 1e-4).

| 조건 | 경험/명목 |
|---|---|
| 이상적 백색 맵 | 0.997 ⟨outputs/verify_cfar.json : white.48x24.rows[89].ratio⟩ |
| 잡음 맵 (Hann + 정합필터) | 1.25 ⟨outputs/verify_cfar.json : chain.NR100.noise.op.rows[89].ratio⟩ |
|   └ Hann 제거 (rect 창) | 0.96 ⟨outputs/verify_cfar.json : control_rect_window_NR100.op.rows[89].ratio⟩ |
|   └ 백색화 정합필터 (거리축 평탄) | 1.25 ⟨outputs/verify_cfar.json : control_whitened_mf_NR100.op.rows[89].ratio⟩ |
|   └ 둘 다 제거 | 1.02 ⟨outputs/verify_cfar.json : control_whitened_mf_rect_NR100.op.rows[89].ratio⟩ |
| 전체 사슬 (직접파 + ECA) | 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩ |

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report04_f5_cause.png", "figure_no": "5", "question": "명목과 경험 사이의 배율을 만드는 것은 무엇인가?", "paper_caption": "Removing the slow-time Hann window and whitening the matched filter returns the empirical false-alarm rate to its nominal value, which identifies training-cell correlation as the origin of the offset.", "vector_pdf": "outputs/figures/report04_f5_cause.pdf", "report": "report04_detector"}-->
![report04_f5_cause.png](outputs/figures/report04_f5_cause.png)

**그림 5.** 명목과 경험 사이의 배율을 만드는 것은 무엇인가?

### 형상 규약 — 교정표가 성립하는 조건

거리창은 ECA 탭 안에 두고, 0-도플러 행 1 ⟨outputs/verify_cfar.json : meta.zd_mask_operational⟩개를 마스킹한다. `check_detector_config()`(`src/passive_process.py:383`)가 두 조건을 검사한다.

| 파형 | 운용 창 | 넓은 창 |
|---|---|---|
| WiFi | 1.53 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.op.rows[89].ratio⟩배 | 41.1 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.wide.rows[89].ratio⟩배 |
| LTE | 2.66 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.op.rows[89].ratio⟩배 | 58.8 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.wide.rows[89].ratio⟩배 |
| 5G | 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배 | 47.7 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.wide.rows[89].ratio⟩배 |

창을 256 ⟨outputs/verify_cfar.json : meta.n_range_wide⟩ 빈으로 넓히면 배율이 두 자릿수가 된다. 교정표는 운용 창 형상에서 측정한 값이다.

### 어느 형상의 교정표가 어디에 쓰이나

형상이 배율을 정한다 — 같은 파형이 운용 창에서 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배, 넓은 창에서 47.70 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.wide.rows[89].ratio⟩배다(바로 위 표). 그래서 명목–경험 관계는 형상마다 다시 잰다.

| 형상 | 무엇을 재나 | 재는 코드 | 그 값을 읽는 곳 |
|---|---|---|---|
| 운용 형상 — CPI 프레임 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩ · `g2x2_t6x6` · 0-도플러 마스크 1 ⟨outputs/verify_cfar.json : meta.zd_mask_operational⟩빈 · 운용 거리창 | 이 편의 교정표 | `benchmark/verify_cfar.py` | `src/experiment_detection.py:358` · `src/experiment_x410.py:175` → 05편 §3 모드별 필요 SNR 표 |
| 자유공간 형상 — 모드별 프레임 수 · 자유공간 거리창 · 0-도플러 가드 | 자유공간 명목 Pfa | `src/freespace_detect.py:711` | `src/experiment_freespace_range.py:206` → 05편 §3 R90 표 |

05편 §3 의 첫 표에 실린 명목 Pfa 는 둘째 줄에서 나온 수다 — 이 편의 교정표와 형상이 달라 값도 다르다.

## §4. 분해능 · 정확도 · 관측가능성

분해능은 두 표적을 가르는 능력이고, 정확도는 한 표적을 찍는 정밀도다. 대역폭이 분해능을, SNR 이 정확도를 정한다.

바이스태틱 규약은 ΔR_b = c/B_ref 다 (R_b = c·τ, 계수 2 없이). 03편 §1 · §2 의 거리 눈금이 같은 정의이고, 아래 표가 같은 값을 싣는다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report04_f6_resolution.png", "figure_no": "6", "question": "대역폭이 정하는 것은 분해능인가 정확도인가?", "paper_caption": "Reference bandwidth sets the bistatic range resolution while signal-to-noise ratio sets the single-target range accuracy, and the two quantities are separated by more than an order of magnitude for every illuminator considered.", "vector_pdf": "outputs/figures/report04_f6_resolution.pdf", "report": "report04_detector"}-->
![report04_f6_resolution.png](outputs/figures/report04_f6_resolution.png)

**그림 6.** 대역폭이 정하는 것은 분해능인가 정확도인가?

### 조명원별 셀 크기와 CRLB

5G 의 상시 기준신호 SSB 는 기준 대역폭 7.20 MHz ⟨outputs/verify_observability.json : cells[2].ref_bw_mhz⟩ 라 셀이 41.64 m ⟨outputs/verify_observability.json : cells[2].drb_bw_m⟩ 다. 같은 반송파에서 PRS 로 가면 98.28 MHz ⟨outputs/verify_observability.json : cells[3].ref_bw_mhz⟩ · 3.05 m ⟨outputs/verify_observability.json : cells[3].drb_bw_m⟩ 가 된다.

도플러 분해능은 Δf_d = 1/T_CPI 다 — T_CPI 0.032 s ⟨outputs/verify_observability.json : cells[0].t_cpi⟩ 에서 31.25 Hz ⟨outputs/verify_observability.json : cells[0].dfd_hz⟩. 그 아래 속도는 §2 의 노치가 먼저 지운다.

| 조명원 / 기준신호 | 기준 대역폭 B_ref | ΔR_b = c/B_ref | 거리 빈 c/f_s | σ_Rb (정확도) | σ_fd |
|---|---|---|---|---|---|
| WiFi80 G1 (VHT-LTF) | 76.56 MHz | 3.92 m | 3.75 m | 0.0162 m | 0.128 Hz |
| LTE20 G1 (CRS) | 17.98 MHz | 16.67 m | 9.76 m | 0.0124 m | 0.022 Hz |
| 5G100 G1 (SSB) | 7.20 MHz | 41.64 m | 2.44 m | 1.4901 m | 1.077 Hz |
| 5G100 G3 (PRS) | 98.28 MHz | 3.05 m | 2.44 m | 0.0073 m | 0.073 Hz |

출처 ⟨outputs/verify_observability.json : cells⟩

ΔR_b 열이 선언 규약이고, 거리 빈은 표본율이 정하는 격자 간격이다. 검출기가 실제로 내는 주엽 폭은 03편 §4 가 이 닫힌형 대비 비율로 싣는다.

### 관측가능성 — 수신기 2대면 위치가 풀린다

TX–RX 기저선 15.07 m ⟨outputs/verify_observability.json : meta.L_m⟩ 형상에서 한 순간의 (R_b, f_d) 는 3차원 위치에 대해 랭크 2 ⟨outputs/verify_observability.json : summary.snapshot_fim_rank⟩ 를 만든다.

기저선을 축으로 표적을 돌리면 R_b 변화가 최대 1.4e-14 m ⟨outputs/verify_observability.json : summary.exact_rotation_max_dRb_m⟩ 다 — 그 방향의 정보량은 SNR 과 관측시간에 무관하게 0 이다. 수신기를 하나 더 놓으면 랭크 6 ⟨outputs/verify_observability.json : summary.fix_2rx_rank⟩ · 위치 RMS 0.19 m ⟨outputs/verify_observability.json : summary.fix_2rx_pos_rms_m⟩ 가 된다.

| 형상 | 유효 랭크 (/6) | 위치 RMS 오차 |
|---|---|---|
| 1RX (baseline) | 3 | 57.75 m |
| 2RX | 6 | 0.19 m |
| 1RX + AoA(1deg) | 6 | 0.12 m |
| 1RX + AoA(5deg) | 6 | 0.60 m |

출처 ⟨outputs/verify_observability.json : fixes⟩

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report04_f7_observability.png", "figure_no": "7", "question": "송수신 한 쌍에 무엇을 더하면 표적 위치가 풀리는가?", "paper_caption": "One transmitter-receiver pair leaves rotation about the baseline unobservable, and a second receiver raises the Fisher information rank to six while reducing the position RMS error to 0.19 m.", "vector_pdf": "outputs/figures/report04_f7_observability.pdf", "report": "report04_detector"}-->
![report04_f7_observability.png](outputs/figures/report04_f7_observability.png)

**그림 7.** 송수신 한 쌍에 무엇을 더하면 표적 위치가 풀리는가?

<!--pk:methods {"kind": "methods", "report": "report04_detector", "text": "All three illuminators are processed by one identical chain. The surveillance channel carries the target echo, the direct-path interference at DNR = 43.0 dB ⟨outputs/verify_eca.json : meta.setups[1].dnr_db⟩ / 60.0 dB ⟨outputs/verify_eca.json : meta.setups[2].dnr_db⟩ / 48.9 dB ⟨outputs/verify_eca.json : meta.setups[0].dnr_db⟩ for WiFi / LTE / 5G NR, static clutter and thermal noise, while the reference channel carries the transmitted frame. A standard extensive cancellation algorithm removes the direct path by a single least-squares projection of the whole CPI onto the subspace spanned by delayed copies of the reference, with n_taps = 24 ⟨outputs/verify_eca.json : meta.setups[1].n_taps⟩ / 14 ⟨outputs/verify_eca.json : meta.setups[2].n_taps⟩ / 32 ⟨outputs/verify_eca.json : meta.setups[0].n_taps⟩. The cross-ambiguity function is then formed frame by frame: a fast-time matched filter against the reference gives one range profile per frame, and a Hann-windowed slow-time FFT over M = 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩ frames gives the Doppler axis, so the Doppler bin is PRF / M and the unambiguous Doppler span is plus or minus PRF / 2. Bistatic range follows R_b = c tau with no factor of two, and the declared range resolution is c / B_ref. Detection uses a two-dimensional cell-averaging CFAR with the g2x2_t6x6 guard and training region (N = 264 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.N_interior⟩ interior training cells) and the 1 ⟨outputs/verify_cfar.json : meta.zd_mask_operational⟩ zero-Doppler row masked; the threshold constant reproduces its analytic value to a relative error of 7.6e-16 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.rel_err⟩. Calibration runs 10,000 ⟨outputs/verify_cfar.json : meta.n_maps_chain⟩ independent range-Doppler maps per waveform at each of nine nominal rates from 1e-06 ⟨outputs/verify_cfar.json : meta.pfa_nominal[8]⟩ to 1e-02 ⟨outputs/verify_cfar.json : meta.pfa_nominal[0]⟩, counts false-alarm cells, and inverts the measured log-log nominal-to-empirical curve to obtain the nominal rate that delivers a target empirical rate; the measurement costs 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ on one GPU and the arithmetic is torch.complex128 throughout.", "tools": ["Python 3.12.13", "PyTorch 2.12.1", "CUDA 13.0"], "params": ["2717 s", "43.0 dB", "48.9 dB", "60.0 dB", "DNR = 43.0", "M = 48", "N = 264", "R_b = c", "n_taps = 24"], "versions": ["CUDA 13.0", "PyTorch 2.12.1", "Python 3.12.13"], "n_words": 296}-->
### 방법 문단 (논문 이관용)

All three illuminators are processed by one identical chain. The surveillance channel carries the target echo, the direct-path interference at DNR = 43.0 dB ⟨outputs/verify_eca.json : meta.setups[1].dnr_db⟩ / 60.0 dB ⟨outputs/verify_eca.json : meta.setups[2].dnr_db⟩ / 48.9 dB ⟨outputs/verify_eca.json : meta.setups[0].dnr_db⟩ for WiFi / LTE / 5G NR, static clutter and thermal noise, while the reference channel carries the transmitted frame. A standard extensive cancellation algorithm removes the direct path by a single least-squares projection of the whole CPI onto the subspace spanned by delayed copies of the reference, with n_taps = 24 ⟨outputs/verify_eca.json : meta.setups[1].n_taps⟩ / 14 ⟨outputs/verify_eca.json : meta.setups[2].n_taps⟩ / 32 ⟨outputs/verify_eca.json : meta.setups[0].n_taps⟩. The cross-ambiguity function is then formed frame by frame: a fast-time matched filter against the reference gives one range profile per frame, and a Hann-windowed slow-time FFT over M = 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩ frames gives the Doppler axis, so the Doppler bin is PRF / M and the unambiguous Doppler span is plus or minus PRF / 2. Bistatic range follows R_b = c tau with no factor of two, and the declared range resolution is c / B_ref. Detection uses a two-dimensional cell-averaging CFAR with the g2x2_t6x6 guard and training region (N = 264 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.N_interior⟩ interior training cells) and the 1 ⟨outputs/verify_cfar.json : meta.zd_mask_operational⟩ zero-Doppler row masked; the threshold constant reproduces its analytic value to a relative error of 7.6e-16 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.rel_err⟩. Calibration runs 10,000 ⟨outputs/verify_cfar.json : meta.n_maps_chain⟩ independent range-Doppler maps per waveform at each of nine nominal rates from 1e-06 ⟨outputs/verify_cfar.json : meta.pfa_nominal[8]⟩ to 1e-02 ⟨outputs/verify_cfar.json : meta.pfa_nominal[0]⟩, counts false-alarm cells, and inverts the measured log-log nominal-to-empirical curve to obtain the nominal rate that delivers a target empirical rate; the measurement costs 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ on one GPU and the arithmetic is torch.complex128 throughout.

버전 — `Python 3.12.13` · `PyTorch 2.12.1` · `CUDA 13.0`

<!--rs:paper-->
<!--pk:defence {"kind": "defence", "report": "report04_detector", "rows": [{"주장": "명목 오경보율과 경험 오경보율은 파형마다 다른 배율로 어긋나고, 그 배율을 재서 CFAR 문턱을 교정했다.", "근거": "그림 4 · `outputs/verify_cfar.json:chain.NR100.dpi_eca.op.rows`", "공격": "그 배율은 CFAR 구현이 틀린 흔적이다.", "답": "문턱 상수는 이론식과 상대오차 7.6e-16 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.rel_err⟩ 안에서 같고, 이상적 백색 맵 500,000 ⟨outputs/verify_cfar.json : meta.n_maps_white⟩장 (셀 564,000,000 ⟨outputs/verify_cfar.json : white.48x24.rows[89].cells⟩개)에서 경험/명목 = 0.997 ⟨outputs/verify_cfar.json : white.48x24.rows[89].ratio⟩ 로 눈금이 1 에 선다."}, {"주장": "교정 없는 세 파형 비교는 서로 다른 실제 오경보율 위에서 이뤄진다.", "근거": "그림 4 · `outputs/verify_cfar.json:chain.LTE20.dpi_eca.calib_op_mask1.points`", "공격": "배율이 두 배 안팎이면 순위가 뒤집힐 만큼 큰가.", "답": "명목 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 에서 실제 오경보율이 WiFi 1.53 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.op.rows[89].ratio⟩배 · LTE 2.66 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.op.rows[89].ratio⟩배 · 5G 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배 로 갈리고, 교정은 명목값을 WiFi 6.27e-05 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ · LTE 2.90e-05 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ · 5G 6.46e-05 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ 로 바꿔 셋을 같은 경험 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 위에 올린다."}, {"주장": "배율의 원인은 slow-time Hann 창이 만드는 도플러축 셀 상관이다.", "근거": "그림 5 · `outputs/verify_cfar.json:control_rect_window_NR100`", "공격": "셀 상관은 어느 검출기에나 있다 — 원인 지목의 근거가 약하다.", "답": "대조군이 확정한다 — Hann 을 rect 창으로 바꾸면 0.96 ⟨outputs/verify_cfar.json : control_rect_window_NR100.op.rows[89].ratio⟩, 백색화 정합필터까지 끄면 1.02 ⟨outputs/verify_cfar.json : control_whitened_mf_rect_NR100.op.rows[89].ratio⟩ 로 눈금이 1 로 돌아온다."}, {"주장": "ECA 소거 깊이의 바닥은 환경이 정한다.", "근거": "그림 2 · `outputs/verify_eca.json:S1_depth_vs_taps`", "공격": "탭이 부족해서 얕게 나온 것이다.", "답": "탭 1~96 스윕에서 깊이가 포화한다 — 직접파만 든 신호에 같은 소거기를 걸면 232.3 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[0].rows[12].depth_dpi_db⟩(float64 한계)까지 내려가고, 측정된 다중경로를 넣으면 56.1 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[0].rows[12].depth_full_db⟩ 에서 멈춘다."}, {"주장": "ECA 는 0-도플러 노치를 대가로 내며, 3 dB 지점은 세 파형이 같다.", "근거": "그림 3 · `outputs/verify_eca.json:S4_target_loss`", "공격": "노치가 드론 속도대를 통째로 먹으면 비교 자체가 무의미하다.", "답": "3 dB 지점은 f_d/Δf_d = 0.596 ⟨outputs/verify_eca.json : S4_target_loss[1].fd_3db_over_dfd⟩ 이고, 프레임 48 ⟨outputs/verify_eca.json : S4_target_loss[4].M⟩개에서 속도 문턱은 WiFi 0.39 m/s ⟨outputs/verify_eca.json : S4_target_loss[4].v_3db_ms⟩ · LTE 1.10 m/s ⟨outputs/verify_eca.json : S4_target_loss[7].v_3db_ms⟩ · 5G 1.16 m/s ⟨outputs/verify_eca.json : S4_target_loss[1].v_3db_ms⟩ 다 — 그 위 속도는 온전히 남는다."}, {"주장": "오경보율 교정은 같은 배경을 수만 번 다시 만드는 통제 시뮬레이션이 한다.", "근거": "§3 · `outputs/prior_census.json:counts.zero_cfar_and_falsealarm`", "공격": "실측 플랫폼이 더 현실적인 근거다.", "답": "census 16 ⟨outputs/prior_census.json : meta.n_papers⟩편 · 전문 198 ⟨outputs/prior_census.json : counts.total_pages⟩쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 ⟨outputs/prior_census.json : counts.zero_cfar_and_falsealarm⟩편이고 검출을 주장한 논문은 1 ⟨outputs/prior_census.json : counts.claims_detection⟩편이다. 실외 실측은 배경을 주어진 대로 받고, 이 편은 맵 10,000 ⟨outputs/verify_cfar.json : meta.n_maps_chain⟩장을 다시 만들어 그 대조를 세운다."}, {"주장": "교정표는 형상마다 다시 잰다.", "근거": "§3 · `src/passive_process.py:383` · `outputs/verify_cfar.json:chain.NR100.dpi_eca.wide`", "공격": "그럼 이 표는 이 형상 하나에서만 쓰는 값이다.", "답": "그렇다 — 같은 파형이 운용 창에서 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배, 넓은 창(256 ⟨outputs/verify_cfar.json : meta.n_range_wide⟩ 빈)에서 47.70 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.wide.rows[89].ratio⟩배다. `check_detector_config()`(`src/passive_process.py:383`)가 형상 조건을 강제하고, 자유공간 형상은 `src/freespace_detect.py:711` 이 다시 잰다."}, {"주장": "송수신 한 쌍의 한 순간 관측량은 3차원 위치에 대해 랭크 2 를 만들고, 수신기 2대가 랭크 6 을 만든다.", "근거": "그림 7 · `outputs/verify_observability.json:summary`", "공격": "그건 기하 문제이고 검출기 성능과 별개다.", "답": "검출 판정은 (R_b, f_d) 셀에서 난다 — 두 양이 위치로 풀리는 조건을 FIM 랭크 2 ⟨outputs/verify_observability.json : summary.snapshot_fim_rank⟩ → 6 ⟨outputs/verify_observability.json : summary.fix_2rx_rank⟩ 로 적어두면 검출 결과가 말하는 범위가 정해진다(2 Rx 위치 RMS 0.19 m ⟨outputs/verify_observability.json : summary.fix_2rx_pos_rms_m⟩)."}]}-->
## 방어선 — 심사자가 때릴 지점과 우리 답

| 주장 | 근거 | 공격 | 답 |
|---|---|---|---|
| 명목 오경보율과 경험 오경보율은 파형마다 다른 배율로 어긋나고, 그 배율을 재서 CFAR 문턱을 교정했다. | 그림 4 · `outputs/verify_cfar.json:chain.NR100.dpi_eca.op.rows` | 그 배율은 CFAR 구현이 틀린 흔적이다. | 문턱 상수는 이론식과 상대오차 7.6e-16 ⟨outputs/verify_cfar.json : alpha_audit.g2x2_t6x6.rel_err⟩ 안에서 같고, 이상적 백색 맵 500,000 ⟨outputs/verify_cfar.json : meta.n_maps_white⟩장 (셀 564,000,000 ⟨outputs/verify_cfar.json : white.48x24.rows[89].cells⟩개)에서 경험/명목 = 0.997 ⟨outputs/verify_cfar.json : white.48x24.rows[89].ratio⟩ 로 눈금이 1 에 선다. |
| 교정 없는 세 파형 비교는 서로 다른 실제 오경보율 위에서 이뤄진다. | 그림 4 · `outputs/verify_cfar.json:chain.LTE20.dpi_eca.calib_op_mask1.points` | 배율이 두 배 안팎이면 순위가 뒤집힐 만큼 큰가. | 명목 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 에서 실제 오경보율이 WiFi 1.53 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.op.rows[89].ratio⟩배 · LTE 2.66 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.op.rows[89].ratio⟩배 · 5G 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배 로 갈리고, 교정은 명목값을 WiFi 6.27e-05 ⟨outputs/verify_cfar.json : chain.WiFi80.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ · LTE 2.90e-05 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ · 5G 6.46e-05 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.calib_op_mask1.points[2].pfa_nominal_needed⟩ 로 바꿔 셋을 같은 경험 1e-04 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].pfa_nom⟩ 위에 올린다. |
| 배율의 원인은 slow-time Hann 창이 만드는 도플러축 셀 상관이다. | 그림 5 · `outputs/verify_cfar.json:control_rect_window_NR100` | 셀 상관은 어느 검출기에나 있다 — 원인 지목의 근거가 약하다. | 대조군이 확정한다 — Hann 을 rect 창으로 바꾸면 0.96 ⟨outputs/verify_cfar.json : control_rect_window_NR100.op.rows[89].ratio⟩, 백색화 정합필터까지 끄면 1.02 ⟨outputs/verify_cfar.json : control_whitened_mf_rect_NR100.op.rows[89].ratio⟩ 로 눈금이 1 로 돌아온다. |
| ECA 소거 깊이의 바닥은 환경이 정한다. | 그림 2 · `outputs/verify_eca.json:S1_depth_vs_taps` | 탭이 부족해서 얕게 나온 것이다. | 탭 1~96 스윕에서 깊이가 포화한다 — 직접파만 든 신호에 같은 소거기를 걸면 232.3 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[0].rows[12].depth_dpi_db⟩(float64 한계)까지 내려가고, 측정된 다중경로를 넣으면 56.1 dB ⟨outputs/verify_eca.json : S1_depth_vs_taps[0].rows[12].depth_full_db⟩ 에서 멈춘다. |
| ECA 는 0-도플러 노치를 대가로 내며, 3 dB 지점은 세 파형이 같다. | 그림 3 · `outputs/verify_eca.json:S4_target_loss` | 노치가 드론 속도대를 통째로 먹으면 비교 자체가 무의미하다. | 3 dB 지점은 f_d/Δf_d = 0.596 ⟨outputs/verify_eca.json : S4_target_loss[1].fd_3db_over_dfd⟩ 이고, 프레임 48 ⟨outputs/verify_eca.json : S4_target_loss[4].M⟩개에서 속도 문턱은 WiFi 0.39 m/s ⟨outputs/verify_eca.json : S4_target_loss[4].v_3db_ms⟩ · LTE 1.10 m/s ⟨outputs/verify_eca.json : S4_target_loss[7].v_3db_ms⟩ · 5G 1.16 m/s ⟨outputs/verify_eca.json : S4_target_loss[1].v_3db_ms⟩ 다 — 그 위 속도는 온전히 남는다. |
| 오경보율 교정은 같은 배경을 수만 번 다시 만드는 통제 시뮬레이션이 한다. | §3 · `outputs/prior_census.json:counts.zero_cfar_and_falsealarm` | 실측 플랫폼이 더 현실적인 근거다. | census 16 ⟨outputs/prior_census.json : meta.n_papers⟩편 · 전문 198 ⟨outputs/prior_census.json : counts.total_pages⟩쪽에서 `CFAR` 와 `false alarm` 이 모두 0회인 논문이 13 ⟨outputs/prior_census.json : counts.zero_cfar_and_falsealarm⟩편이고 검출을 주장한 논문은 1 ⟨outputs/prior_census.json : counts.claims_detection⟩편이다. 실외 실측은 배경을 주어진 대로 받고, 이 편은 맵 10,000 ⟨outputs/verify_cfar.json : meta.n_maps_chain⟩장을 다시 만들어 그 대조를 세운다. |
| 교정표는 형상마다 다시 잰다. | §3 · `src/passive_process.py:383` · `outputs/verify_cfar.json:chain.NR100.dpi_eca.wide` | 그럼 이 표는 이 형상 하나에서만 쓰는 값이다. | 그렇다 — 같은 파형이 운용 창에서 1.52 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.op.rows[89].ratio⟩배, 넓은 창(256 ⟨outputs/verify_cfar.json : meta.n_range_wide⟩ 빈)에서 47.70 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.wide.rows[89].ratio⟩배다. `check_detector_config()`(`src/passive_process.py:383`)가 형상 조건을 강제하고, 자유공간 형상은 `src/freespace_detect.py:711` 이 다시 잰다. |
| 송수신 한 쌍의 한 순간 관측량은 3차원 위치에 대해 랭크 2 를 만들고, 수신기 2대가 랭크 6 을 만든다. | 그림 7 · `outputs/verify_observability.json:summary` | 그건 기하 문제이고 검출기 성능과 별개다. | 검출 판정은 (R_b, f_d) 셀에서 난다 — 두 양이 위치로 풀리는 조건을 FIM 랭크 2 ⟨outputs/verify_observability.json : summary.snapshot_fim_rank⟩ → 6 ⟨outputs/verify_observability.json : summary.fix_2rx_rank⟩ 로 적어두면 검출 결과가 말하는 범위가 정해진다(2 Rx 위치 RMS 0.19 m ⟨outputs/verify_observability.json : summary.fix_2rx_pos_rms_m⟩). |

### 인용 (논문 형식 · 게재상태 포함)

1. F. Colone, C. Palmarini, T. Martelli, E. Tilli, "Sliding Extensive Cancellation Algorithm for Disturbance Removal in Passive Radar", IEEE Transactions on Aerospace and Electronic Systems 52(3):1309-1326, 2016 [게재] (ECA-S. 이 편의 제거단은 CPI 1회 최소제곱 사영의 standard ECA 다 — src/passive_process.py:13)<!--pk:cite {"kind": "cite", "authors": "F. Colone, C. Palmarini, T. Martelli, E. Tilli", "title": "Sliding Extensive Cancellation Algorithm for Disturbance Removal in Passive Radar", "venue": "IEEE Transactions on Aerospace and Electronic Systems", "volume": "52(3)", "pages": "1309-1326", "year": 2016, "status": "published", "status_ko": "게재", "arxiv": null, "doi": null, "note": "ECA-S. 이 편의 제거단은 CPI 1회 최소제곱 사영의 standard ECA 다 — src/passive_process.py:13", "text": "F. Colone, C. Palmarini, T. Martelli, E. Tilli, \"Sliding Extensive Cancellation Algorithm for Disturbance Removal in Passive Radar\", IEEE Transactions on Aerospace and Electronic Systems 52(3):1309-1326, 2016 [게재] (ECA-S. 이 편의 제거단은 CPI 1회 최소제곱 사영의 standard ECA 다 — src/passive_process.py:13)"}-->
2. Z. Zhou et al., "OpenISAC: An Open-Source Real-Time Experimentation Platform for OFDM-ISAC", arXiv preprint, 2026 [프리프린트, arXiv:2601.03535v2] (전문에서 CFAR · false alarm · detection probability 0회)<!--pk:cite {"kind": "cite", "authors": "Z. Zhou et al.", "title": "OpenISAC: An Open-Source Real-Time Experimentation Platform for OFDM-ISAC", "venue": "arXiv preprint", "volume": null, "pages": null, "year": 2026, "status": "preprint", "status_ko": "프리프린트", "arxiv": "2601.03535v2", "doi": null, "note": "전문에서 CFAR · false alarm · detection probability 0회", "text": "Z. Zhou et al., \"OpenISAC: An Open-Source Real-Time Experimentation Platform for OFDM-ISAC\", arXiv preprint, 2026 [프리프린트, arXiv:2601.03535v2] (전문에서 CFAR · false alarm · detection probability 0회)"}-->
3. R. Liu et al., "Clutter-Aware Integrated Sensing and Communication: Models, Methods, and Future Directions", Proceedings of the IEEE 114(1):52-91, 2026 [게재] (census 에서 CFAR 를 쓴 게재 논문)<!--pk:cite {"kind": "cite", "authors": "R. Liu et al.", "title": "Clutter-Aware Integrated Sensing and Communication: Models, Methods, and Future Directions", "venue": "Proceedings of the IEEE", "volume": "114(1)", "pages": "52-91", "year": 2026, "status": "published", "status_ko": "게재", "arxiv": null, "doi": null, "note": "census 에서 CFAR 를 쓴 게재 논문", "text": "R. Liu et al., \"Clutter-Aware Integrated Sensing and Communication: Models, Methods, and Future Directions\", Proceedings of the IEEE 114(1):52-91, 2026 [게재] (census 에서 CFAR 를 쓴 게재 논문)"}-->
4. H. Liu et al., "DMSNet: Cross-Band Learning for Multi-Target Sensing in Multi-Band ISAC", arXiv preprint, 2026 [프리프린트, arXiv:2607.17655v1] (census 에서 CFAR 빈도가 가장 높은 프리프린트)<!--pk:cite {"kind": "cite", "authors": "H. Liu et al.", "title": "DMSNet: Cross-Band Learning for Multi-Target Sensing in Multi-Band ISAC", "venue": "arXiv preprint", "volume": null, "pages": null, "year": 2026, "status": "preprint", "status_ko": "프리프린트", "arxiv": "2607.17655v1", "doi": null, "note": "census 에서 CFAR 빈도가 가장 높은 프리프린트", "text": "H. Liu et al., \"DMSNet: Cross-Band Learning for Multi-Target Sensing in Multi-Band ISAC\", arXiv preprint, 2026 [프리프린트, arXiv:2607.17655v1] (census 에서 CFAR 빈도가 가장 높은 프리프린트)"}-->
5. Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski, "Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band", NATO STO-MP-MSG-SET-183, paper 13, 2021 [게재] (패시브 WiFi 드론 검출의 종단 산출물 — FDTD RCS → 커버리지 → 50 m OTA)<!--pk:cite {"kind": "cite", "authors": "Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski", "title": "Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band", "venue": "NATO STO-MP-MSG-SET-183, paper 13", "volume": null, "pages": null, "year": 2021, "status": "published", "status_ko": "게재", "arxiv": null, "doi": null, "note": "패시브 WiFi 드론 검출의 종단 산출물 — FDTD RCS → 커버리지 → 50 m OTA", "text": "Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski, \"Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band\", NATO STO-MP-MSG-SET-183, paper 13, 2021 [게재] (패시브 WiFi 드론 검출의 종단 산출물 — FDTD RCS → 커버리지 → 50 m OTA)"}-->

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 실외 클러터 배경 위에서 같은 Pfa 스윕을 돌린다 | 교정 배율이 배경에 따라 얼마나 움직이는지 수치로 확정된다 | `benchmark/verify_cfar.py` · 06편 §2 측정 조건 |
| 맵 수를 한 자릿수 올려 명목 1e-06 ⟨outputs/verify_cfar.json : chain.NR100.dpi_eca.calib_op_mask1.points[4].pfa_target_emp⟩ 구간까지 측정한다 | 저 Pfa 운용점의 교정값이 측정 구간 안으로 들어온다 | `benchmark/verify_cfar.py --maps` · `calib_op_mask1.points` |
| 훈련셀에서도 0-도플러 행을 빼는 CFAR 변형을 만든다 | 넓은 창에서 마스크 폭 3 이 만드는 배율 0.65 ⟨outputs/verify_cfar.json : chain.LTE20.dpi_eca.wide.rows[90].ratio⟩배가 마스크 폭과 분리된다 | `src/passive_process.py:352` |
| 수신기 2대 형상으로 검출 실험을 재설계한다 | 위치 RMS 0.19 m ⟨outputs/verify_observability.json : summary.fix_2rx_pos_rms_m⟩ 가 검출 실험에서 확인된다 | 05편 검출 결과 · §4 표의 2RX 행 |
| 표적 σ 를 리포트 02 의 앵커 위에서 읽는다 | Pd 절대값이 교정된 Pfa 와 같은 근거 위에 선다 | 05편 검출 결과 |
| 회전 블레이드 산란을 별도 검증으로 세운다 | 마이크로도플러를 검출기에 넣는 조건이 결정된다 | future work |